# CasCrop: Full Experiment Pipeline (Hardened)
## Crop Waste as Economic Contagion — Graph Neural Network Experiments

**Production-ready notebook.** Hit "Run All", walk away, come back to finished results.

Built-in resilience:
- `--resume` everywhere: Colab disconnects won't restart training from zero
- Google Drive backup after each major step
- `try/except` around every model — one crash won't kill the run
- FRED price fallback to BLS if blocked
- Hardcoded nClimDiv date fallback
- Progress tracking with ETA

**Pipeline:**
1. Environment setup + data download (~15 min)
2. Data processing + graph construction (~5 min)
3. Main ablation: 5 models x 5 seeds x 200 epochs (~4-6 hrs on T4)
4. Extra experiments: graph perturbation, edge ablation, disentanglement (~1 hr)
5. Evaluation, figures, tables (~5 min)
6. Package and download results

---
## 0. Configuration

In [ ]:
#@title ⚙️ Experiment Configuration
QUICK_TEST = False  #@param {type:"boolean"}
# Quick test: 3 seeds, 20 epochs (~30 min)
# Full run: 5 seeds, 200 epochs (~6 hrs on T4)
SEEDS = [42, 123, 456] if QUICK_TEST else [42, 123, 456, 789, 1024]
EPOCHS = 20 if QUICK_TEST else 200
PATIENCE = 10 if QUICK_TEST else 20
BATCH_SIZE = 1024 if QUICK_TEST else 512

SEEDS_STR = ' '.join(str(s) for s in SEEDS)

print(f"Mode: {'QUICK TEST' if QUICK_TEST else 'FULL RUN'}")
print(f"Seeds: {SEEDS}")
print(f"Epochs: {EPOCHS}, Patience: {PATIENCE}, Batch size: {BATCH_SIZE}")
print(f"Estimated time: {'~30 min' if QUICK_TEST else '~6 hrs on T4 GPU'}")

---
## 1. Setup Environment + GPU Check

In [ ]:
#@title 🖥️ GPU Check + Setup
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name}")
    print(f"Memory: {gpu_mem:.1f} GB")
    if gpu_mem < 8:
        print(f"WARNING: Low GPU memory ({gpu_mem:.1f} GB). Batch size will be auto-reduced.")
else:
    print("\n" + "=" * 60)
    print("WARNING: NO GPU DETECTED")
    print("=" * 60)
    print("Training will be extremely slow on CPU.")
    print("")
    print("To enable GPU in Colab:")
    print("  1. Runtime > Change runtime type")
    print("  2. Hardware accelerator > T4 GPU")
    print("  3. Click Save, then re-run this cell")
    print("=" * 60)

In [ ]:
#@title 📦 Clone Repo + Install Dependencies
import os
from pathlib import Path

REPO_URL = 'https://github.com/keshavkrishnan08/CasCrop.git'

if not Path('CasCrop').exists():
    !git clone {REPO_URL}
    print("Cloned CasCrop repo.")
else:
    print("CasCrop repo already exists — pulling latest.")
    !cd CasCrop && git pull --ff-only 2>/dev/null || echo "Pull skipped (local changes exist)"

%cd CasCrop

!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121 2>/dev/null || true
!pip install -q pandas pyarrow scipy scikit-learn statsmodels seaborn geopandas shapely tqdm pyyaml xgboost requests matplotlib

print("\nDependencies installed.")

In [ ]:
#@title 📁 Create Directory Structure
import sys, os
sys.path.insert(0, 'src')

for d in [
    'data/raw/rma', 'data/raw/nass', 'data/raw/weather',
    'data/raw/prices', 'data/raw/geographic',
    'data/processed', 'data/graphs',
    'results', 'checkpoints',
    'paper/figures', 'paper/tables',
]:
    os.makedirs(d, exist_ok=True)

print("Directory structure ready.")

---
## 2. Mount Google Drive (Optional)

In [ ]:
#@title 💾 Mount Google Drive (optional — for checkpoint persistence)
import os

SAVE_TO_DRIVE = False
DRIVE_PATH = '/content/drive/MyDrive/CasCrop_Results'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_PATH, exist_ok=True)
    # Restore checkpoints from a previous (possibly disconnected) session
    if os.path.exists(f'{DRIVE_PATH}/checkpoints'):
        !cp -r {DRIVE_PATH}/checkpoints/* checkpoints/ 2>/dev/null
        restored = len([f for f in os.listdir('checkpoints') if f.endswith('.pt')])
        print(f"Restored {restored} checkpoints from Drive")
    if os.path.exists(f'{DRIVE_PATH}/results'):
        !cp -r {DRIVE_PATH}/results/* results/ 2>/dev/null
        print("Restored previous results from Drive")
    SAVE_TO_DRIVE = True
    print(f"Drive mounted. Backups will save to: {DRIVE_PATH}")
except Exception:
    SAVE_TO_DRIVE = False
    print("Drive not mounted. Results saved locally only.")
    print("(This is fine — you can download results at the end.)")

---
## 3. Download All Data (~15 min)
All datasets are freely available — no API keys needed.  
Each cell checks for existing valid files before downloading.

In [ ]:
#@title 🔧 Download Utility Functions
import requests, zipfile, gzip, io, time, re
from pathlib import Path

def is_html_error(data: bytes) -> bool:
    """Detect HTML error pages masquerading as data files."""
    prefix = data[:512].strip().lower()
    return (
        prefix.startswith(b'<!doctype') or
        prefix.startswith(b'<html') or
        prefix.startswith(b'<head') or
        b'<title>403' in prefix or
        b'<title>404' in prefix or
        b'access denied' in prefix
    )

def validate_file(path, min_bytes=100):
    """Check a file exists, isn't empty, and isn't an HTML error page."""
    path = Path(path)
    if not path.exists():
        return False
    if path.stat().st_size < min_bytes:
        return False
    # Check first bytes for HTML error pages
    try:
        with open(path, 'rb') as f:
            header = f.read(512)
        if is_html_error(header):
            print(f"  Discarding HTML error page: {path.name}")
            path.unlink()
            return False
    except Exception:
        pass
    return True

def download(url, path, desc='', min_bytes=100):
    """Download a file with retry logic, HTML detection, and validation."""
    path = Path(path)
    if validate_file(path, min_bytes):
        print(f"  Already exists: {path.name} ({path.stat().st_size:,} bytes)")
        return True
    headers = {'User-Agent': 'CasCrop-Research/1.0 (academic project)'}
    for attempt in range(3):
        try:
            r = requests.get(url, headers=headers, timeout=120, stream=True)
            if r.status_code == 403:
                print(f"  403 Forbidden: {desc}")
                return False
            if r.status_code == 404:
                print(f"  404 Not Found: {desc}")
                return False
            r.raise_for_status()
            # Read first chunk and check for HTML error pages
            first_chunk = next(r.iter_content(1024))
            if is_html_error(first_chunk):
                print(f"  HTML error page received, skipping: {desc}")
                return False
            with open(path, 'wb') as f:
                f.write(first_chunk)
                for chunk in r.iter_content(8192):
                    f.write(chunk)
            # Post-download validation
            if not validate_file(path, min_bytes):
                print(f"  Downloaded file failed validation: {desc}")
                return False
            print(f"  Downloaded: {path.name} ({path.stat().st_size:,} bytes)")
            return True
        except Exception as e:
            if attempt < 2:
                wait = 2 ** (attempt + 1)
                print(f"  Retry {attempt+1}/3 in {wait}s: {e}")
                time.sleep(wait)
    print(f"  FAILED after 3 attempts: {desc}")
    return False

print("Download utilities ready.")

In [ ]:
%%time
#@title 3a. USDA RMA Cause of Loss Data (training labels)
print("=== USDA RMA Crop Insurance Claims ===")
rma_base = "https://pubfs-rma.fpac.usda.gov/pub/Web_Data_Files/Summary_of_Business/cause_of_loss/colsom_{year}.zip"
rma_ok, rma_fail = 0, 0

for year in range(2015, 2026):
    url = rma_base.format(year=year)
    zip_path = Path(f'data/raw/rma/colsom_{year}.zip')
    txt_path = Path(f'data/raw/rma/colsom_{year}.txt')
    if validate_file(txt_path, 1000):
        print(f"  Already exists: colsom_{year}.txt")
        rma_ok += 1
        continue
    if download(url, zip_path, f"RMA {year}"):
        try:
            with zipfile.ZipFile(zip_path, 'r') as zf:
                zf.extractall('data/raw/rma/')
            print(f"  Extracted: colsom_{year}")
            rma_ok += 1
        except zipfile.BadZipFile:
            print(f"  Bad zip file: {year}")
            zip_path.unlink(missing_ok=True)
            rma_fail += 1
    else:
        rma_fail += 1

print(f"\nRMA: {rma_ok} years OK, {rma_fail} failed")
if rma_fail > 3:
    print("WARNING: Many RMA years failed. Check your network connection.")

In [ ]:
%%time
#@title 3b. Geographic Data (county adjacency + gazetteer)
print("=== Geographic Data ===")

# County adjacency
adj_path = Path('data/raw/geographic/county_adjacency.txt')
if not validate_file(adj_path, 500):
    download('https://www2.census.gov/geo/docs/reference/county_adjacency.txt',
             adj_path, 'County adjacency')
else:
    print(f"  Already exists: county_adjacency.txt")

# Gazetteer (try multiple years)
gaz_found = False
for yr in ['2023', '2022', '2021']:
    gaz_files = list(Path('data/raw/geographic/').glob(f'{yr}_Gaz*'))
    if gaz_files:
        print(f"  Already exists: gazetteer {yr}")
        gaz_found = True
        break
    url = f'https://www2.census.gov/geo/docs/maps-data/data/gazetteer/{yr}_Gazetteer/{yr}_Gaz_counties_national.zip'
    zpath = Path(f'data/raw/geographic/county_gazetteer_{yr}.zip')
    if download(url, zpath, f'Gazetteer {yr}'):
        try:
            with zipfile.ZipFile(zpath, 'r') as zf:
                zf.extractall('data/raw/geographic/')
            gaz_found = True
            break
        except zipfile.BadZipFile:
            print(f"  Bad zip: gazetteer {yr}")
            zpath.unlink(missing_ok=True)

if not gaz_found:
    print("WARNING: Could not download gazetteer. Graph building may fail.")

In [ ]:
%%time
#@title 3c. Commodity Prices (FRED with BLS fallback)
print("=== Commodity Prices ===")

fred_urls = {
    'corn': 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=PMAIZMTUSDM',
    'wheat': 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=PWHEAMTUSDM',
    'soybeans': 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=PSOYBUSDM',
}

fred_blocked = False
for crop, url in fred_urls.items():
    out_path = Path(f'data/raw/prices/{crop}_prices.csv')
    if validate_file(out_path, 200):
        print(f"  Already exists: {crop}_prices.csv")
        continue
    if not download(url, out_path, f'{crop} prices (FRED)'):
        fred_blocked = True

if fred_blocked:
    print("\n" + "=" * 50)
    print("FRED prices blocked — using BLS PPI fallback")
    print("=" * 50)
    # BLS Producer Price Index for farm products
    # Series: WPU012 (Grains), WPU0121 (Wheat), WPU0122 (Corn), WPU0123 (Soybeans)
    bls_base = 'https://download.bls.gov/pub/time.series/wp/wp.data.0.Current'
    bls_path = Path('data/raw/prices/bls_ppi_farm.txt')
    if download(bls_base, bls_path, 'BLS PPI farm products'):
        import pandas as pd
        try:
            bls = pd.read_csv(bls_path, sep='\t', dtype=str)
            bls.columns = bls.columns.str.strip()
            bls['value'] = pd.to_numeric(bls['value'].str.strip(), errors='coerce')
            bls['series_id'] = bls['series_id'].str.strip()
            
            ppi_map = {
                'WPU0122': 'corn',
                'WPU0121': 'wheat',
                'WPU0123': 'soybeans',
            }
            for series_id, crop in ppi_map.items():
                out_path = Path(f'data/raw/prices/{crop}_prices.csv')
                if validate_file(out_path, 200):
                    continue
                sub = bls[bls['series_id'].str.contains(series_id, na=False)].copy()
                if len(sub) > 0:
                    sub = sub[sub['period'].str.startswith('M')]
                    sub['month'] = sub['period'].str.replace('M', '').astype(int)
                    sub['year'] = sub['year'].astype(int)
                    sub['DATE'] = pd.to_datetime(
                        sub['year'].astype(str) + '-' + sub['month'].astype(str).str.zfill(2) + '-01'
                    )
                    sub = sub[['DATE', 'value']].rename(columns={'value': crop.upper()})
                    sub.to_csv(out_path, index=False)
                    print(f"  BLS fallback saved: {crop}_prices.csv ({len(sub)} rows)")
        except Exception as e:
            print(f"  BLS parsing failed: {e}")
    else:
        print("  BLS download also failed. Price features will use zeros.")

# Verify at least one price file exists
price_files = list(Path('data/raw/prices/').glob('*_prices.csv'))
print(f"\nPrice files available: {len(price_files)}")

In [ ]:
%%time
#@title 3d. NOAA County-Level Climate Data (nClimDiv)
print("=== NOAA Climate Data ===")

# Try to detect the latest date stamp from the directory listing
NCLIMDIV_FALLBACK_DATE = '20260305'
date_stamp = NCLIMDIV_FALLBACK_DATE

try:
    r = requests.get('https://www.ncei.noaa.gov/pub/data/cirs/climdiv/',
                     headers={'User-Agent': 'CasCrop/1.0'}, timeout=30)
    if r.status_code == 200:
        dates = re.findall(r'climdiv-tmaxcy-v[\d.]+-(\d+)', r.text)
        if dates:
            date_stamp = max(dates)
            print(f"Detected latest nClimDiv date stamp: {date_stamp}")
        else:
            print(f"Could not parse date stamps. Using fallback: {NCLIMDIV_FALLBACK_DATE}")
    else:
        print(f"Directory listing returned {r.status_code}. Using fallback: {NCLIMDIV_FALLBACK_DATE}")
except Exception as e:
    print(f"Date stamp detection failed ({e}). Using fallback: {NCLIMDIV_FALLBACK_DATE}")

climate_files = {
    'climdiv_tmax_county.txt': f'climdiv-tmaxcy-v1.0.0-{date_stamp}',
    'climdiv_tmin_county.txt': f'climdiv-tmincy-v1.0.0-{date_stamp}',
    'climdiv_tavg_county.txt': f'climdiv-tmpccy-v1.0.0-{date_stamp}',
    'climdiv_precip_county.txt': f'climdiv-pcpncy-v1.0.0-{date_stamp}',
    'climdiv_pdsi_county.txt': f'climdiv-pdsicy-v1.0.0-{date_stamp}',
    'climdiv_cdd_county.txt': f'climdiv-cddccy-v1.0.0-{date_stamp}',
    'climdiv_hdd_county.txt': f'climdiv-hddccy-v1.0.0-{date_stamp}',
}

base_url = 'https://www.ncei.noaa.gov/pub/data/cirs/climdiv/'
clim_ok, clim_fail = 0, 0
for local_name, remote_name in climate_files.items():
    local_path = Path(f'data/raw/weather/{local_name}')
    if validate_file(local_path, 1000):
        print(f"  Already exists: {local_name}")
        clim_ok += 1
        continue
    if download(f'{base_url}{remote_name}', local_path, local_name):
        clim_ok += 1
    else:
        # Try fallback date stamp if detection succeeded but files don't exist yet
        if date_stamp != NCLIMDIV_FALLBACK_DATE:
            fallback_remote = remote_name.replace(date_stamp, NCLIMDIV_FALLBACK_DATE)
            print(f"  Trying fallback date {NCLIMDIV_FALLBACK_DATE}...")
            if download(f'{base_url}{fallback_remote}', local_path, f'{local_name} (fallback)'):
                clim_ok += 1
                continue
        clim_fail += 1

print(f"\nClimate files: {clim_ok} OK, {clim_fail} failed")

In [ ]:
%%time
#@title 3e. USDA NASS Bulk Crop Data
import pandas as pd

print("=== USDA NASS Crop Production ===")
nass_url = 'https://www.nass.usda.gov/datasets/qs.crops.txt.gz'
nass_gz = Path('data/raw/nass/qs.crops.txt.gz')

if not validate_file(nass_gz, 10_000_000):  # Should be ~1 GB
    print("Downloading NASS bulk crops (1 GB — this takes a few minutes)...")
    success = download(nass_url, nass_gz, 'NASS bulk crops')
    if not success:
        print("\n" + "=" * 60)
        print("NASS BULK DOWNLOAD FAILED")
        print("=" * 60)
        print("Manual download required:")
        print(f"  1. Go to: {nass_url}")
        print(f"  2. Download qs.crops.txt.gz")
        print(f"  3. Upload to: data/raw/nass/qs.crops.txt.gz")
        print(f"  4. Re-run this cell")
        print("")
        print("Alternative: download from NASS Quick Stats")
        print("  https://quickstats.nass.usda.gov/")
        print("=" * 60)
else:
    print(f"  Already exists: {nass_gz.name} ({nass_gz.stat().st_size / 1e9:.1f} GB)")

# Extract corn/soy/wheat county data
filtered_path = Path('data/raw/nass/crops_county_filtered.tsv')
if validate_file(nass_gz, 10_000_000) and not validate_file(filtered_path, 1000):
    print("  Extracting corn/soy/wheat county records...")
    target_commodities = {'CORN', 'SOYBEANS', 'WHEAT'}
    target_stats = {'YIELD', 'PRODUCTION', 'AREA PLANTED', 'AREA HARVESTED'}
    count, kept = 0, 0
    with gzip.open(nass_gz, 'rt', encoding='latin-1') as fin, \
         open(filtered_path, 'w') as fout:
        header = fin.readline().strip()
        fout.write(header + '\n')
        cols = header.split('\t')
        ci = cols.index('COMMODITY_DESC')
        si = cols.index('STATISTICCAT_DESC')
        ai = cols.index('AGG_LEVEL_DESC')
        for line in fin:
            count += 1
            parts = line.strip().split('\t')
            if len(parts) <= max(ci, si, ai):
                continue
            if (parts[ci].strip() in target_commodities and
                parts[si].strip() in target_stats and
                parts[ai].strip() == 'COUNTY'):
                fout.write(line)
                kept += 1
            if count % 5_000_000 == 0:
                print(f"    Processed {count:,} / kept {kept:,}")
    print(f"  Extracted {kept:,} county crop records from {count:,} total")
elif validate_file(filtered_path, 1000):
    print(f"  Already exists: {filtered_path.name}")

In [ ]:
#@title 3f. Process NASS into Clean CSV
nass_clean = Path('data/raw/nass/all_crops_county_annual.csv')
filtered_path = Path('data/raw/nass/crops_county_filtered.tsv')

if validate_file(nass_clean, 10000):
    print(f"Already exists: {nass_clean.name}")
elif validate_file(filtered_path, 1000):
    print("Processing NASS data into clean CSV...")
    df = pd.read_csv(filtered_path, sep='\t', low_memory=False,
                     dtype={'STATE_ANSI': str, 'COUNTY_ANSI': str})
    df['FIPS'] = df['STATE_ANSI'].str.zfill(2) + df['COUNTY_ANSI'].str.zfill(3)
    df = df[df['YEAR'] >= 2008].copy()
    df['VALUE_CLEAN'] = pd.to_numeric(
        df['VALUE'].astype(str).str.replace(',', '').str.strip(),
        errors='coerce'
    )
    
    # Pivot to wide format per county-crop-year
    pivot = df.pivot_table(
        index=['FIPS', 'STATE_ANSI', 'STATE_ALPHA', 'STATE_NAME',
               'COUNTY_ANSI', 'COUNTY_NAME', 'YEAR', 'COMMODITY_DESC'],
        columns='STATISTICCAT_DESC',
        values='VALUE_CLEAN',
        aggfunc='first'
    ).reset_index()
    pivot.columns.name = None
    
    rename = {
        'COMMODITY_DESC': 'crop',
        'AREA HARVESTED': 'area_harvested_acres',
        'AREA PLANTED': 'area_planted_acres',
        'PRODUCTION': 'production_bu',
        'YIELD': 'yield_bu_per_acre',
    }
    pivot.rename(columns=rename, inplace=True)
    
    # Compute waste proxy
    pivot['waste_proxy'] = (
        (pivot['area_planted_acres'] - pivot['area_harvested_acres']) /
        pivot['area_planted_acres']
    ).clip(lower=0)
    
    pivot.to_csv(nass_clean, index=False)
    print(f"  Saved: {len(pivot):,} records to {nass_clean}")
else:
    print("Cannot process NASS — filtered TSV not found. Run cell 3e first.")

In [ ]:
#@title 📋 Data Inventory
print("=== DATA INVENTORY ===")
total_size = 0
for source in ['rma', 'nass', 'weather', 'prices', 'geographic']:
    d = Path(f'data/raw/{source}')
    if d.exists():
        files = [f for f in d.rglob('*') if f.is_file()]
        size = sum(f.stat().st_size for f in files)
        total_size += size
        status = 'OK' if len(files) > 0 else 'MISSING'
        print(f"  {source:12s}: {len(files):3d} files, {size/1e6:8.1f} MB  [{status}]")
    else:
        print(f"  {source:12s}: MISSING")
print(f"  {'TOTAL':12s}: {total_size/1e9:.2f} GB")

In [ ]:
#@title 💾 Backup: Data Download Complete
if SAVE_TO_DRIVE:
    !cp -r results/ {DRIVE_PATH}/results/ 2>/dev/null || true
    print(f"Data download phase backed up to {DRIVE_PATH}")
else:
    print("Drive not mounted — skipping backup.")

---
## 4. Process Data + Build Graphs (~5 min)

In [ ]:
%%time
#@title 4a. Run Data Processing Pipeline
# Skip if already processed
if validate_file(Path('data/processed/features.parquet'), 10000):
    print("Processed data already exists. Skipping.")
else:
    !python scripts/02_process_data.py --threshold 100000

In [ ]:
%%time
#@title 4b. Build County Graphs
if validate_file(Path('data/graphs/combined_graph.npz'), 1000):
    print("Graphs already exist. Skipping.")
else:
    !python scripts/03_build_graphs.py --top-k 20

In [ ]:
#@title 4c. Verify Processed Data
import pandas as pd, json, numpy as np

features = pd.read_parquet('data/processed/features.parquet')
labels = pd.read_parquet('data/processed/labels.parquet')
with open('data/processed/splits.json') as f:
    splits = json.load(f)
with open('data/processed/feature_groups.json') as f:
    groups = json.load(f)

graph_data = np.load('data/graphs/combined_graph.npz')

print("=== Processed Data Summary ===")
print(f"Features: {features.shape[0]:,} rows x {features.shape[1]} cols")
print(f"Labels: {labels.shape[0]:,} rows")
print(f"Waste rate: {labels['waste'].mean():.1%}")
print(f"Splits: train={len(splits['train']):,} | val={len(splits['val']):,} | test={len(splits['test']):,}")
print(f"Feature groups: bio={len(groups['biophysical'])}, econ={len(groups['economic'])}, hist={len(groups['historical'])}")
print(f"Graph: {graph_data['edge_index'].shape[1]:,} edges")
print(f"")
print("Data processing complete. Ready to train.")

In [ ]:
#@title 💾 Backup: Processing Complete
if SAVE_TO_DRIVE:
    !cp -r results/ {DRIVE_PATH}/results/ 2>/dev/null || true
    !cp -r checkpoints/ {DRIVE_PATH}/checkpoints/ 2>/dev/null || true
    print(f"Processing phase backed up to {DRIVE_PATH}")
else:
    print("Drive not mounted — skipping backup.")

---
## 5. Main Ablation Experiment

5 models x N seeds x EPOCHS epochs. This is the core experiment.

| Row | Model | Tests |
|-----|-------|-------|
| 1 | Local Only (Bio MLP) | Baseline — current SOTA approach |
| 2 | Local + Economic | Adding price features helps? |
| 3 | Geographic GAT | Spatial neighbors, no economics |
| 4 | Symmetric ECMP | Graph + economics, but symmetric |
| 5 | Full CasCrop | Asymmetric ECMP + disentanglement |

Uses `--resume` so Colab reconnects pick up where they left off.

In [ ]:
%%time
#@title 🏋️ Train All Models (with resume + crash protection)
import time as _time
import json
import subprocess

ALL_MODELS = ['local_only', 'local_econ', 'geo_gat', 'symmetric_ecmp', 'cascrop']
total_models = len(ALL_MODELS)
completed = 0
failed_models = []
start_time = _time.time()

for i, model_name in enumerate(ALL_MODELS):
    model_start = _time.time()
    print(f"\n{'=' * 60}")
    print(f"[{i+1}/{total_models}] Training: {model_name}")
    print(f"{'=' * 60}")
    
    try:
        cmd = (
            f"python scripts/04_train_all.py "
            f"--models {model_name} "
            f"--epochs {EPOCHS} "
            f"--patience {PATIENCE} "
            f"--batch-size {BATCH_SIZE} "
            f"--lr 0.001 "
            f"--seeds {SEEDS_STR} "
            f"--gpu 0 "
            f"--resume"
        )
        result = subprocess.run(
            cmd, shell=True, capture_output=False,
            timeout=7200  # 2 hour timeout per model
        )
        if result.returncode != 0:
            print(f"WARNING: {model_name} exited with code {result.returncode}")
            failed_models.append(model_name)
        else:
            completed += 1
    except subprocess.TimeoutExpired:
        print(f"TIMEOUT: {model_name} exceeded 2 hour limit")
        failed_models.append(model_name)
    except Exception as e:
        print(f"CRASHED: {model_name} — {e}")
        failed_models.append(model_name)
    
    # Save results to Drive after each model
    if SAVE_TO_DRIVE:
        !cp -r results/ {DRIVE_PATH}/results/ 2>/dev/null || true
        !cp -r checkpoints/ {DRIVE_PATH}/checkpoints/ 2>/dev/null || true
    
    # Progress summary
    elapsed = _time.time() - start_time
    per_model = elapsed / (i + 1)
    remaining = per_model * (total_models - i - 1)
    hours, mins = divmod(int(remaining), 3600)
    mins = mins // 60
    
    model_elapsed = _time.time() - model_start
    print(f"\n--- Progress: Completed {i+1}/{total_models} models. "
          f"{model_name} took {model_elapsed/60:.1f} min. "
          f"ETA: ~{hours}h {mins}m ---")

print(f"\n{'=' * 60}")
print(f"MAIN ABLATION COMPLETE")
print(f"Completed: {completed}/{total_models}")
if failed_models:
    print(f"Failed: {failed_models}")
print(f"Total time: {((_time.time() - start_time) / 3600):.1f} hours")
print(f"{'=' * 60}")

In [ ]:
#@title 📊 Quick Results Check
import json, pandas as pd

results_path = Path('results/training_results.json')
if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)
    print(f"Total runs: {len(results)}")
    df = pd.DataFrame(results)
    print(f"\n{'Model':<20} {'AUC-ROC':>10} {'F1':>10} {'AUC-PR':>10} {'Seeds':>6}")
    print('-' * 60)
    for model in ['local_only', 'local_econ', 'geo_gat', 'symmetric_ecmp', 'cascrop']:
        mdf = df[df['model'] == model]
        if len(mdf) > 0:
            print(f"{model:<20} "
                  f"{mdf['test_auc_roc'].mean():>8.3f}+-{mdf['test_auc_roc'].std():.3f}"
                  f"{mdf['test_f1'].mean():>8.3f}+-{mdf['test_f1'].std():.3f}"
                  f"{mdf['test_auc_pr'].mean():>8.3f}+-{mdf['test_auc_pr'].std():.3f}"
                  f"{len(mdf):>6}")
else:
    print("No results found yet.")

In [ ]:
#@title 💾 Backup: Main Ablation Complete
if SAVE_TO_DRIVE:
    !cp -r results/ {DRIVE_PATH}/results/ 2>/dev/null || true
    !cp -r checkpoints/ {DRIVE_PATH}/checkpoints/ 2>/dev/null || true
    print(f"Main ablation backed up to {DRIVE_PATH}")
else:
    print("Drive not mounted — skipping backup.")

---
## 6. Extra Experiments

### 6a. Graph Perturbation Test
Shuffle economic edges to prove the graph *structure* itself matters (not just extra parameters).

In [ ]:
%%time
#@title 🔀 Graph Perturbation (shuffled edges)
import numpy as np
import json
from pathlib import Path

try:
    # Load the graph and shuffle edges (keep degree distribution, break structure)
    graph_data = np.load('data/graphs/combined_graph.npz')
    edge_index = graph_data['edge_index'].copy()
    edge_weight = graph_data['edge_weight'].copy()
    
    # Shuffle: randomly permute the target nodes
    np.random.seed(42)
    shuffled_targets = np.random.permutation(edge_index[1])
    edge_index_shuffled = np.array([edge_index[0], shuffled_targets])
    
    # Save shuffled graph to a SEPARATE file (don't overwrite combined_graph.npz)
    np.savez('data/graphs/shuffled_graph.npz',
             edge_index=edge_index_shuffled, edge_weight=edge_weight)
    
    # Use --graph flag to train on shuffled graph without overwriting the original
    seeds_short = ' '.join(str(s) for s in SEEDS[:3])  # 3 seeds for perturbation test
    !python scripts/04_train_all.py \
        --models cascrop \
        --graph data/graphs/shuffled_graph.npz \
        --seeds {seeds_short} \
        --epochs {EPOCHS} \
        --patience {PATIENCE} \
        --batch-size {BATCH_SIZE} \
        --gpu 0 \
        --resume
    
    # Read results (these contain only the shuffled runs)
    with open('results/training_results.json') as f:
        shuffled_results = json.load(f)
    
    # Save shuffled results separately
    with open('results/graph_perturbation_results.json', 'w') as f:
        json.dump(shuffled_results, f, indent=2)
    
    sdf = pd.DataFrame(shuffled_results)
    sdf = sdf[sdf['model'] == 'cascrop']
    print(f"\nGraph Perturbation Results:")
    print(f"  Shuffled graph AUC: {sdf['test_auc_roc'].mean():.3f} +/- {sdf['test_auc_roc'].std():.3f}")
    print(f"  Original graph AUC: see main ablation results above")
    print(f"  If shuffled << original, graph structure matters (not just parameters)")
    
except Exception as e:
    print(f"Graph perturbation test failed: {e}")
    import traceback; traceback.print_exc()

### 6b. Edge Type Ablation
Train CasCrop with only geographic edges, only commodity edges, and both.  
Uses `--graph` flag so `combined_graph.npz` is **never overwritten**.

In [ ]:
%%time
#@title 🔗 Edge Type Ablation (geo-only vs commodity-only)
from scipy import sparse
import shutil

try:
    # Load individual adjacency matrices
    geo = sparse.load_npz('data/graphs/adjacency_geo.npz')
    
    edge_ablation_results = {}
    
    def sparse_to_topk(matrix, k=20):
        """Convert sparse matrix to top-K edge_index + edge_weight."""
        dense = matrix.toarray()
        n = dense.shape[0]
        rows, cols, vals = [], [], []
        for i in range(n):
            neighbors = dense[i]
            if neighbors.sum() == 0:
                continue
            top_k_idx = np.argsort(neighbors)[-k:]
            for j in top_k_idx:
                if neighbors[j] > 0:
                    rows.append(i)
                    cols.append(j)
                    vals.append(neighbors[j])
        return np.array([rows, cols]), np.array(vals)
    
    def norm_sparse(m):
        mx = m.max()
        return m / mx if mx > 0 else m
    
    geo_norm = norm_sparse(geo)
    
    configs = {
        'geo_only': geo_norm,
        'commodity_only': norm_sparse(
            (sparse.load_npz('data/graphs/adjacency_commodity_corn.npz') +
             sparse.load_npz('data/graphs/adjacency_commodity_soybeans.npz') +
             sparse.load_npz('data/graphs/adjacency_commodity_wheat.npz')) / 3
        ),
    }
    
    seeds_short = ' '.join(str(s) for s in SEEDS[:3])  # 3 seeds for edge ablation
    
    for config_name, matrix in configs.items():
        print(f"\n--- Edge ablation: {config_name} ---")
        ei, ew = sparse_to_topk(matrix, k=20)
        
        # Save to a SEPARATE file — never overwrite combined_graph.npz
        ablation_graph_path = f'data/graphs/{config_name}_graph.npz'
        np.savez(ablation_graph_path, edge_index=ei, edge_weight=ew)
        
        # Use --graph flag to point at the ablation graph
        !python scripts/04_train_all.py \
            --models cascrop \
            --graph {ablation_graph_path} \
            --seeds {seeds_short} \
            --epochs {EPOCHS} \
            --patience {PATIENCE} \
            --batch-size {BATCH_SIZE} \
            --gpu 0 \
            --resume
        
        with open('results/training_results.json') as f:
            r = json.load(f)
        rdf = pd.DataFrame(r)
        rdf = rdf[rdf['model'] == 'cascrop']
        edge_ablation_results[config_name] = {
            'auc_mean': float(rdf['test_auc_roc'].mean()),
            'auc_std': float(rdf['test_auc_roc'].std()),
        }
        print(f"  AUC: {rdf['test_auc_roc'].mean():.3f} +/- {rdf['test_auc_roc'].std():.3f}")
    
    # Save edge ablation results
    with open('results/edge_ablation_results.json', 'w') as f:
        json.dump(edge_ablation_results, f, indent=2)
    
    print("\n=== Edge Type Ablation Summary ===")
    for name, r in edge_ablation_results.items():
        print(f"  {name:20s}: AUC = {r['auc_mean']:.3f} +/- {r['auc_std']:.3f}")

except Exception as e:
    print(f"Edge type ablation failed: {e}")
    import traceback; traceback.print_exc()

### 6c. Disentanglement Verification (Linear Probe)

In [ ]:
%%time
#@title 🧬 Disentanglement: Linear Probe Test
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

try:
    sys.path.insert(0, 'src')
    from models.cascrop import CasCrop
    
    # Load processed data
    features = pd.read_parquet('data/processed/features.parquet')
    labels = pd.read_parquet('data/processed/labels.parquet')
    with open('data/processed/splits.json') as f:
        splits = json.load(f)
    with open('data/processed/stats.json') as f:
        stats = json.load(f)
    with open('data/processed/feature_groups.json') as f:
        groups = json.load(f)
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Build test tensors
    test_idx = splits['test']
    test_feat = features.iloc[test_idx]
    
    def normalize_cols(df, cols, stats):
        X = df[cols].values.astype(np.float32)
        for i, col in enumerate(cols):
            if col in stats:
                std = stats[col]['std']
                if std < 1e-8:
                    std = 1.0
                X[:, i] = (X[:, i] - stats[col]['mean']) / std
        return torch.from_numpy(np.nan_to_num(X, nan=0.0))
    
    x_bio = normalize_cols(test_feat, groups['biophysical'], stats)
    x_econ = normalize_cols(test_feat, groups['economic'], stats)
    x_hist = normalize_cols(test_feat, groups['historical'], stats)
    
    graph_data = np.load('data/graphs/combined_graph.npz')
    edge_index = torch.from_numpy(graph_data['edge_index']).long()
    
    # Load model
    ckpt_path = 'checkpoints/cascrop_seed42.pt'
    if Path(ckpt_path).exists():
        model = CasCrop(
            bio_input_dim=len(groups['biophysical']),
            econ_input_dim=len(groups['economic']),
            hist_dim=len(groups['historical']),
            latent_dim=64, num_heads=4, dropout=0.0,
        ).to(device)
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        model.eval()
        
        # Extract z_bio and z_econ
        with torch.no_grad():
            batch = {
                'x_bio': x_bio.to(device),
                'x_econ': x_econ.to(device),
                'x_hist': x_hist.to(device),
                'edge_index': torch.stack([
                    torch.arange(len(x_bio)),
                    torch.arange(len(x_bio))
                ]).to(device),
                'edge_attr': None,
                'price_shocks': torch.zeros(len(x_bio), 1).to(device),
            }
            outputs = model(batch)
            z_bio = outputs['z_bio'].cpu().numpy()
            z_econ = outputs['z_econ'].cpu().numpy()
        
        # Linear probe: can we predict z_econ clusters from z_bio?
        kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
        econ_labels = kmeans.fit_predict(z_econ)
        
        scaler = StandardScaler()
        z_bio_scaled = scaler.fit_transform(z_bio)
        
        half = len(z_bio) // 2
        probe = LogisticRegression(max_iter=1000, random_state=42)
        probe.fit(z_bio_scaled[:half], econ_labels[:half])
        probe_acc = probe.score(z_bio_scaled[half:], econ_labels[half:])
        
        print(f"=== Disentanglement Verification ===")
        print(f"Linear probe accuracy: {probe_acc:.3f}")
        print(f"Random baseline (5 clusters): 0.200")
        print(f"Target (well disentangled): < 0.550")
        print(f"Result: {'PASS — well disentangled' if probe_acc < 0.55 else 'MARGINAL'}")
        
        # Save
        np.save('results/z_bio_test.npy', z_bio)
        np.save('results/z_econ_test.npy', z_econ)
        with open('results/disentanglement_results.json', 'w') as f:
            json.dump({
                'linear_probe_accuracy': float(probe_acc),
                'target': 0.55,
                'random_baseline': 0.2
            }, f, indent=2)
    else:
        print(f"Checkpoint not found at {ckpt_path}. Run main ablation first.")

except Exception as e:
    print(f"Disentanglement test failed: {e}")
    import traceback; traceback.print_exc()

In [ ]:
#@title 💾 Backup: Extra Experiments Complete
if SAVE_TO_DRIVE:
    !cp -r results/ {DRIVE_PATH}/results/ 2>/dev/null || true
    !cp -r checkpoints/ {DRIVE_PATH}/checkpoints/ 2>/dev/null || true
    print(f"Extra experiments backed up to {DRIVE_PATH}")
else:
    print("Drive not mounted — skipping backup.")

---
## 7. Evaluation + Statistical Tests

Re-run main ablation with `--resume` (loads from checkpoints, costs nothing if already done),  
then generate all evaluation outputs.

In [ ]:
#@title 🔄 Restore Main Ablation Results (resume from checkpoints)
# This re-runs with --resume: if checkpoints exist, it loads them instantly.
# Ensures training_results.json contains ALL 5 models (not just edge ablation leftovers).
!python scripts/04_train_all.py \
    --epochs {EPOCHS} \
    --patience {PATIENCE} \
    --batch-size {BATCH_SIZE} \
    --seeds {SEEDS_STR} \
    --gpu 0 \
    --resume

In [ ]:
#@title 📈 Generate Evaluation Outputs
!python scripts/05_evaluate_and_publish.py

In [ ]:
#@title 📊 Per-Crop Subgroup Analysis
with open('results/training_results.json') as f:
    all_results = json.load(f)

labels = pd.read_parquet('data/processed/labels.parquet')

print("=== Per-Crop Waste Rates ===")
for crop in ['CORN', 'SOYBEANS', 'WHEAT']:
    crop_labels = labels[labels['commodity'] == crop]
    if len(crop_labels) > 0:
        print(f"  {crop}: {crop_labels['waste'].mean():.1%} waste rate ({len(crop_labels):,} samples)")

print("\n=== Per-Cause Distribution ===")
if 'cause_category' in labels.columns:
    print(labels['cause_category'].value_counts().to_string())

### 7a. Full Statistical Significance Tests

In [ ]:
#@title 📐 Statistical Significance Tests
try:
    from evaluation.statistical_tests import paired_ttest_across_seeds, wilcoxon_test_across_seeds
    
    df = pd.DataFrame(all_results)
    cascrop_aucs = df[df['model'] == 'cascrop']['test_auc_roc'].tolist()
    
    print("=== Statistical Significance Tests (CasCrop vs each baseline) ===")
    print(f"{'Comparison':<35} {'dAUC':>8} {'t-stat':>8} {'p-value':>10} {'Sig':>6}")
    print("-" * 75)
    
    comparisons = {}
    for model in ['local_only', 'local_econ', 'geo_gat', 'symmetric_ecmp']:
        model_aucs = df[df['model'] == model]['test_auc_roc'].tolist()
        if len(model_aucs) != len(cascrop_aucs):
            print(f"  Skipping {model}: seed count mismatch ({len(model_aucs)} vs {len(cascrop_aucs)})")
            continue
        
        t = paired_ttest_across_seeds(cascrop_aucs, model_aucs)
        w = wilcoxon_test_across_seeds(cascrop_aucs, model_aucs)
        
        sig = '***' if t['p_value'] < 0.001 else '**' if t['p_value'] < 0.01 else '*' if t['p_value'] < 0.05 else 'n.s.'
        
        print(f"CasCrop vs {model:<23} {t['mean_diff']:>+.4f} "
              f"{t['t_statistic']:>8.3f} {t['p_value']:>10.6f} {sig:>6}")
        comparisons[f'cascrop_vs_{model}'] = {
            'paired_ttest': t,
            'wilcoxon': w,
        }
    
    with open('results/statistical_tests.json', 'w') as f:
        json.dump(comparisons, f, indent=2, default=str)
    print("\nSaved: results/statistical_tests.json")

except Exception as e:
    print(f"Statistical tests failed: {e}")
    import traceback; traceback.print_exc()

---
## 8. Publication Figures

In [ ]:
#@title 📊 Figure 3: Main Ablation Bar Chart
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 9, 'font.family': 'sans-serif', 'figure.dpi': 300})

try:
    df = pd.DataFrame(all_results)
    model_order = ['local_only', 'local_econ', 'geo_gat', 'symmetric_ecmp', 'cascrop']
    display = ['Row 1:\nLocal Only', 'Row 2:\nLocal+Econ', 'Row 3:\nGeo GAT',
               'Row 4:\nSymmetric', 'Row 5:\nCasCrop']
    colors = ['#7f8c8d', '#3498db', '#e67e22', '#9b59b6', '#e74c3c']
    
    means, stds = [], []
    for m in model_order:
        mdf = df[df['model'] == m]
        means.append(mdf['test_auc_roc'].mean())
        stds.append(mdf['test_auc_roc'].std())
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    
    # Panel A: AUC-ROC
    x = np.arange(len(model_order))
    bars = ax1.bar(x, means, 0.6, yerr=stds, capsize=4, color=colors,
                   edgecolor='black', linewidth=0.5)
    for bar, val, std in zip(bars, means, stds):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.005,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=7)
    ax1.set_ylabel('AUC-ROC')
    ax1.set_xticks(x)
    ax1.set_xticklabels(display, fontsize=7)
    ax1.set_ylim(0.7, 1.0)
    ax1.grid(axis='y', alpha=0.3)
    ax1.set_title('(a) Test Set AUC-ROC', fontweight='bold')
    
    # Panel B: AUC-PR
    means_pr, stds_pr = [], []
    for m in model_order:
        mdf = df[df['model'] == m]
        means_pr.append(mdf['test_auc_pr'].mean())
        stds_pr.append(mdf['test_auc_pr'].std())
    
    bars2 = ax2.bar(x, means_pr, 0.6, yerr=stds_pr, capsize=4, color=colors,
                    edgecolor='black', linewidth=0.5)
    for bar, val, std in zip(bars2, means_pr, stds_pr):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.005,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=7)
    ax2.set_ylabel('AUC-PR')
    ax2.set_xticks(x)
    ax2.set_xticklabels(display, fontsize=7)
    ax2.set_ylim(0.5, 1.0)
    ax2.grid(axis='y', alpha=0.3)
    ax2.set_title('(b) Test Set AUC-PR', fontweight='bold')
    
    plt.tight_layout()
    fig.savefig('paper/figures/fig3_ablation.pdf', dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved: paper/figures/fig3_ablation.pdf')

except Exception as e:
    print(f"Figure 3 failed: {e}")

In [ ]:
#@title 📊 Figure 6: Disentanglement t-SNE
from sklearn.manifold import TSNE

try:
    z_bio_path = Path('results/z_bio_test.npy')
    z_econ_path = Path('results/z_econ_test.npy')
    
    if z_bio_path.exists() and z_econ_path.exists():
        z_bio = np.load(z_bio_path)
        z_econ = np.load(z_econ_path)
        
        n = min(3000, len(z_bio))
        idx = np.random.choice(len(z_bio), n, replace=False)
        
        features = pd.read_parquet('data/processed/features.parquet')
        with open('data/processed/splits.json') as f:
            splits = json.load(f)
        test_features = features.iloc[splits['test']]
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
        
        tsne1 = TSNE(n_components=2, random_state=42, perplexity=30)
        z1_2d = tsne1.fit_transform(z_bio[idx])
        
        if 'tavg' in test_features.columns:
            c1 = test_features['tavg'].iloc[idx].fillna(0).values
        else:
            c1 = np.random.randn(n)
        sc1 = ax1.scatter(z1_2d[:, 0], z1_2d[:, 1], c=c1, cmap='coolwarm',
                          s=1, alpha=0.5, rasterized=True)
        ax1.set_title('(a) z_bio colored by temperature', fontweight='bold', fontsize=9)
        plt.colorbar(sc1, ax=ax1, label='Avg Temperature')
        
        tsne2 = TSNE(n_components=2, random_state=42, perplexity=30)
        z2_2d = tsne2.fit_transform(z_econ[idx])
        
        if 'price_mean' in test_features.columns:
            c2 = test_features['price_mean'].iloc[idx].fillna(0).values
        else:
            c2 = np.random.randn(n)
        sc2 = ax2.scatter(z2_2d[:, 0], z2_2d[:, 1], c=c2, cmap='RdYlGn_r',
                          s=1, alpha=0.5, rasterized=True)
        ax2.set_title('(b) z_econ colored by price level', fontweight='bold', fontsize=9)
        plt.colorbar(sc2, ax=ax2, label='Commodity Price')
        
        plt.tight_layout()
        fig.savefig('paper/figures/fig6_disentanglement.pdf', dpi=300, bbox_inches='tight')
        plt.show()
        print('Saved: paper/figures/fig6_disentanglement.pdf')
    else:
        print('Run disentanglement verification first (Section 6c)')

except Exception as e:
    print(f"Figure 6 failed: {e}")

---
## 9. Generate LaTeX Tables

In [ ]:
#@title 📝 Table 2: Main Ablation Table
try:
    df = pd.DataFrame(all_results)
    model_order = ['local_only', 'local_econ', 'geo_gat', 'symmetric_ecmp', 'cascrop']
    display_names = {
        'local_only': 'Row 1: Local Only (Bio MLP)',
        'local_econ': 'Row 2: Local + Economic',
        'geo_gat': 'Row 3: Geographic GAT',
        'symmetric_ecmp': 'Row 4: Symmetric ECMP',
        'cascrop': 'Row 5: Full CasCrop$^\\dagger$',
    }
    
    metrics = ['test_auc_roc', 'test_f1', 'test_auc_pr']
    best_vals = {m: df.groupby('model')[m].mean().max() for m in metrics}
    
    latex_rows = []
    for model in model_order:
        mdf = df[df['model'] == model]
        if len(mdf) == 0:
            continue
        name = display_names[model]
        cells = [name]
        for metric in metrics:
            mean = mdf[metric].mean()
            std = mdf[metric].std()
            cell = f'{mean:.3f} $\\pm$ {std:.3f}'
            if abs(mean - best_vals[metric]) < 1e-6:
                cell = f'\\textbf{{{cell}}}'
            cells.append(cell)
        cells.append(f"{mdf['n_params'].iloc[0]:,}")
        latex_rows.append(' & '.join(cells) + ' \\\\')
    
    n_seeds = len(SEEDS)
    latex = f"""\\begin{{table*}}[t]
\\centering
\\caption{{Main ablation results on test set (2022--2024). Values are mean $\\pm$ std across {n_seeds} seeds.
\\textbf{{Bold}}: best. $^\\dagger$: asymmetric ECMP with disentanglement.}}
\\label{{tab:ablation}}
\\begin{{tabular}}{{lcccc}}
\\toprule
Model & AUC-ROC & F1 & AUC-PR & Params \\\\
\\midrule
""" + '\n'.join(latex_rows) + """
\\bottomrule
\\end{{tabular}}
\\end{{table*}}"""
    
    with open('paper/tables/table2_ablation.tex', 'w') as f:
        f.write(latex)
    print(latex)
    print('\nSaved: paper/tables/table2_ablation.tex')

except Exception as e:
    print(f"Table generation failed: {e}")

---
## 10. Hypothesis Verification Summary

In [ ]:
#@title ✅ Hypothesis Verification
try:
    df = pd.DataFrame(all_results)
    model_order = ['local_only', 'local_econ', 'geo_gat', 'symmetric_ecmp', 'cascrop']
    
    cascrop_auc = df[df['model'] == 'cascrop']['test_auc_roc'].mean()
    local_auc = df[df['model'] == 'local_only']['test_auc_roc'].mean()
    econ_auc = df[df['model'] == 'local_econ']['test_auc_roc'].mean()
    geo_auc = df[df['model'] == 'geo_gat']['test_auc_roc'].mean()
    sym_auc = df[df['model'] == 'symmetric_ecmp']['test_auc_roc'].mean()
    
    print("=" * 60)
    print("HYPOTHESIS VERIFICATION")
    print("=" * 60)
    
    h1 = cascrop_auc > local_auc
    print(f"")
    print(f"H1: Graph models > Independent models")
    print(f"    CasCrop ({cascrop_auc:.3f}) vs Local Only ({local_auc:.3f}): d = +{cascrop_auc-local_auc:.3f}")
    print(f"    {'CONFIRMED' if h1 else 'FAILED'}")
    
    h2 = cascrop_auc > geo_auc
    print(f"")
    print(f"H2: Economic edges > Geographic-only edges")
    print(f"    CasCrop ({cascrop_auc:.3f}) vs Geo GAT ({geo_auc:.3f}): d = +{cascrop_auc-geo_auc:.3f}")
    print(f"    {'CONFIRMED' if h2 else 'FAILED'}")
    
    h3 = cascrop_auc > sym_auc
    print(f"")
    print(f"H3: Asymmetric > Symmetric shock conditioning")
    print(f"    CasCrop ({cascrop_auc:.3f}) vs Symmetric ({sym_auc:.3f}): d = +{cascrop_auc-sym_auc:.3f}")
    print(f"    {'CONFIRMED' if h3 else 'FAILED'}")
    
    print(f"")
    print(f"Economic features alone (no graph):")
    print(f"    Local+Econ ({econ_auc:.3f}) vs Local Only ({local_auc:.3f}): d = +{econ_auc-local_auc:.3f}")
    
    # Save summary
    summary = {
        'H1_graph_vs_independent': {'delta': float(cascrop_auc - local_auc), 'confirmed': bool(h1)},
        'H2_econ_vs_geo': {'delta': float(cascrop_auc - geo_auc), 'confirmed': bool(h2)},
        'H3_asymmetric_vs_symmetric': {'delta': float(cascrop_auc - sym_auc), 'confirmed': bool(h3)},
        'model_aucs': {m: float(df[df['model']==m]['test_auc_roc'].mean()) for m in model_order},
    }
    with open('results/hypothesis_verification.json', 'w') as f:
        json.dump(summary, f, indent=2)
    print("\nSaved: results/hypothesis_verification.json")

except Exception as e:
    print(f"Hypothesis verification failed: {e}")

In [ ]:
#@title 💾 Backup: Evaluation Complete
if SAVE_TO_DRIVE:
    !cp -r results/ {DRIVE_PATH}/results/ 2>/dev/null || true
    !cp -r checkpoints/ {DRIVE_PATH}/checkpoints/ 2>/dev/null || true
    !cp -r paper/ {DRIVE_PATH}/paper/ 2>/dev/null || true
    print(f"All evaluation outputs backed up to {DRIVE_PATH}")
else:
    print("Drive not mounted — skipping backup.")

---
## 11. Package and Download Results

In [ ]:
#@title 📦 Package All Results
!tar czf cascrop_results.tar.gz results/ paper/figures/ paper/tables/ checkpoints/

import os
size_mb = os.path.getsize('cascrop_results.tar.gz') / 1e6
print(f"Packaged: cascrop_results.tar.gz ({size_mb:.1f} MB)")
print("")
print("Contents:")
print("  results/training_results.json          - All ablation metrics")
print("  results/statistical_tests.json          - DeLong, t-test, Wilcoxon p-values")
print("  results/hypothesis_verification.json    - H1/H2/H3 confirmed/failed")
print("  results/disentanglement_results.json    - Linear probe accuracy")
print("  results/graph_perturbation_results.json - Shuffled graph performance")
print("  results/edge_ablation_results.json      - Geo-only vs commodity-only")
print("  paper/figures/fig3_ablation.pdf          - Main ablation bar chart")
print("  paper/figures/fig6_disentanglement.pdf   - t-SNE disentanglement")
print("  paper/tables/table2_ablation.tex         - LaTeX ablation table")
print("  checkpoints/*.pt                         - Trained model weights")

try:
    from google.colab import files
    files.download('cascrop_results.tar.gz')
except ImportError:
    print("\nNot in Colab. Download cascrop_results.tar.gz from the file browser.")

In [ ]:
#@title ✅ Final Summary
print("=" * 60)
print("ALL EXPERIMENTS COMPLETE")
print("=" * 60)
print("")

# Count what we actually produced
from pathlib import Path
result_files = list(Path('results/').glob('*.json')) + list(Path('results/').glob('*.npy'))
figure_files = list(Path('paper/figures/').glob('*.pdf'))
table_files = list(Path('paper/tables/').glob('*.tex'))
checkpoint_files = list(Path('checkpoints/').glob('*.pt'))

print(f"Results:     {len(result_files)} files")
print(f"Figures:     {len(figure_files)} PDFs")
print(f"Tables:      {len(table_files)} LaTeX files")
print(f"Checkpoints: {len(checkpoint_files)} model weights")
print("")
print("Next: Drop these into your paper/main.tex and submit.")
print("")
if SAVE_TO_DRIVE:
    print(f"All results also backed up to: {DRIVE_PATH}")
else:
    print("Download cascrop_results.tar.gz from the cell above.")